# 🤖 Indoor Object Detection: Free Cloud GPU Training (Kaggle)

Train a custom, ground-level **YOLOv8 / YOLOv11** model for autonomous **Indoor Object Detection** using Kaggle's free GPU quota (30 GPU-hours/week).

### ⚙️ One-Time Kaggle Settings (Right Sidebar):
1. **Session Options → Accelerator:** Select **GPU T4 x 2** (or GPU P100).
2. **Session Options → Internet:** Toggle to **ON** (required to download base weights).
3. **Input → Add Input:** Upload your Roboflow exported dataset zip as a Kaggle Dataset and attach it.

In [1]:
# Step 1: Verify Cloud GPU
!nvidia-smi

Wed Sep 16 09:53:58 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   50C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Step 2: Install Ultralytics & Core Dependencies
!pip install --quiet ultralytics opencv-python onnx

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 6.5 MB/s eta 0:00:00


In [3]:
# Step 3: Copy Unpacked Dataset to Working Directory (Bulletproof)
import os
import glob
import shutil

# 1. Hunt down the yaml file anywhere in the input folder
yaml_files = glob.glob('/kaggle/input/**/data.yaml', recursive=True)

if not yaml_files:
    print("❌ ERROR: Could not find the dataset. Did you attach it on the right sidebar?")
else:
    source_dir = os.path.dirname(yaml_files[0])
    print(f"📦 Found dataset hidden at: {source_dir}")
    
    # 2. Copy everything safely to the writable working directory
    shutil.copytree(source_dir, '/kaggle/working/', dirs_exist_ok=True)
    print("✅ Dataset successfully copied to /kaggle/working/!")
    # 3. Rewrite data.yaml to enforce absolute Kaggle path (Bulletproof)
    yaml_path = '/kaggle/working/data.yaml'
    with open(yaml_path, 'r') as f:
        content = f.read()
    import re
    if 'path:' in content:
        content = re.sub(r'path:.*', 'path: /kaggle/working', content)
    else:
        content = 'path: /kaggle/working\n' + content
    with open(yaml_path, 'w') as f:
        f.write(content)
    print('✅ data.yaml path rewritten for Kaggle!')


📦 Found dataset hidden at: /kaggle/input/datasets/aungmyopaing/everyday-household-objects-dataset/data/dataset_v4
✅ Dataset successfully copied to /kaggle/working/!
✅ data.yaml path rewritten for Kaggle!


In [4]:
# Step 4: Fine-Tune YOLO on Cloud GPU (Indoor Detection)
from ultralytics import YOLO

# Load Nano base model (ultra-fast for edge devices)
model = YOLO('yolov8n.pt')

# Train for 100 epochs on GPU
results = model.train(
    data='/kaggle/working/data.yaml',
    epochs=100,
    imgsz=416,
    batch=32,      # Pushed to 32 to utilize the T4 GPUs
    device=0,
    optimizer='AdamW',
    save=True,
    project='/kaggle/working/runs',
    name='indoor_detection_v1'
)
print('🎉 Training Complete!')

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.153 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, fr

/usr/local/lib/python3.12/dist-packages/ray/train/_internal/session.py:676: UserWarning: `get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.
  warnings.warn(


      2/100      2.89G      1.621      1.852      1.556         75        416: 100% ━━━━━━━━━━━━ 391/391 6.5it/s 1:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 4.0it/s 12.2s
                   all       3119      14718      0.486      0.108      0.055     0.0233

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      3/100      2.91G      1.538      1.716      1.493         90        416: 100% ━━━━━━━━━━━━ 391/391 6.4it/s 1:01
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 49/49 3.9it/s 12.5s
                   all       3119      14718      0.776     0.0934     0.0797     0.0374

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      4/100      2.91G      1.496      1.625      1.456         41        416: 100% ━━━━━━━━━━━━ 391/391 6.5it/s 59.7s
                 Class     Images  Instances      Box

In [5]:
# Step 5: Export to ONNX for Low-Latency Deployment
best_weights = '/kaggle/working/runs/indoor_detection_v1/weights/best.pt'
trained_model = YOLO(best_weights)
trained_model.export(format='onnx', imgsz=416, simplify=True)
print('✅ Exported to ONNX: best.onnx')

Ultralytics 8.4.153 🚀 Python-3.12.13 torch-2.10.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 72 layers, 3,009,548 parameters, 0 gradients, 3.4 GFLOPs

PyTorch: starting from '/kaggle/working/runs/indoor_detection_v1/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 24, 3549) (5.9 MB)
requirements: Ultralytics requirements ['onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 216ms
 Downloaded onnxruntime
Prepared 2 packages in 338ms
Installed 2 packages in 15ms
 + onnxruntime==1.30.0
 + onnxslim==0.1.96

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 18...
ONNX: slimming with onnxslim 0.1.96...
O

In [6]:
# Step 6: Package Model Weights for Download
# Step 7: Package Entire Run Results for Download
import shutil

# This will zip EVERYTHING inside the runs folder!
shutil.make_archive('/kaggle/working/indoor_detection_results', 'zip', '/kaggle/working/runs')

print('✅ Output Zip Created: /kaggle/working/indoor_detection_results.zip')

✅ Output Zip Created: /kaggle/working/indoor_detection_results.zip


In [7]:
# Step 7: Package Entire Run Results for Download
import shutil

# This will zip EVERYTHING inside the runs folder!
shutil.make_archive('/kaggle/working/indoor_detection_results', 'zip', '/kaggle/working/runs')

print('✅ Output Zip Created: /kaggle/working/indoor_detection_results.zip')

✅ Output Zip Created: /kaggle/working/indoor_detection_results.zip
